In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# 1. Configuração do dispositivo de execução (GPU/CPU)
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if dispositivo.type == "cuda":
    print(f"Treino acelerado por GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Treino a executar na CPU.")

# 2. Pipeline de transformação para tensores
transformacao = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 3. Carregamento do CIFAR-10
print("A carregar CIFAR-10...")
treino_set = torchvision.datasets.CIFAR10(
    root="./dados_cifar", train=True, download=True, transform=transformacao
)
teste_set = torchvision.datasets.CIFAR10(
    root="./dados_cifar", train=False, download=True, transform=transformacao
)

treino_loader = torch.utils.data.DataLoader(treino_set, batch_size=64, shuffle=True)
teste_loader = torch.utils.data.DataLoader(teste_set, batch_size=64, shuffle=False)

classes = ('aviao', 'carro', 'passaro', 'gato', 'veado',
           'cao', 'sapo', 'cavalo', 'navio', 'caminhao')

# 4. Arquitetura da CNN Básica
class CNNBasica(nn.Module):
    def __init__(self):
        super(CNNBasica, self).__init__()
        # Bloco Convolucional 1: Entrada 3 canais (RGB) -> Saída 16 filtros 3x3
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 32x32 -> 16x16
        
        # Bloco Convolucional 2: Entrada 16 filtros -> Saída 32 filtros 3x3
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 16x16 -> 8x8
        
        # Classificador Linear
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Achatamento (Flatten)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

modelo = CNNBasica().to(dispositivo)
criterio = nn.CrossEntropyLoss()
otimizador = optim.Adam(modelo.parameters(), lr=0.001)

# -------------------------------------------------------------------------
# Função de validação no conjunto de teste
# -------------------------------------------------------------------------
def avaliar_acuracia(modelo, loader, dispositivo):
    modelo.eval()
    acertos = 0
    total = 0
    with torch.no_grad():
        for imagens, rotulos in loader:
            imagens, rotulos = imagens.to(dispositivo), rotulos.to(dispositivo)
            saidas = modelo(imagens)
            _, preditos = torch.max(saidas, 1)
            total += rotulos.size(0)
            acertos += (preditos == rotulos).sum().item()
    modelo.train()
    return (100.0 * acertos) / total

# 5. Treino com Paragem Antecipada (Early Stopping)
MAX_EPOCAS = 15
LIMIAR_ACURACIA = 65.0

print(f"\n--- INICIANDO TREINO (Máximo: {MAX_EPOCAS} épocas | Meta: >={LIMIAR_ACURACIA}%) ---")

for epoca in range(1, MAX_EPOCAS + 1):
    modelo.train()
    perda_total_epoca = 0.0
    
    # Itera por todos os 782 lotes da época atual
    for imagens, rotulos in treino_loader:
        imagens, rotulos = imagens.to(dispositivo), rotulos.to(dispositivo)
        
        otimizador.zero_grad()
        predicoes = modelo(imagens)
        perda = criterio(predicoes, rotulos)
        perda.backward()
        otimizador.step()
        
        perda_total_epoca += perda.item()

    perda_media = perda_total_epoca / len(treino_loader)
    
    # Avaliação de teste executada uma única vez ao final da época completa
    acuracia_atual = avaliar_acuracia(modelo, teste_loader, dispositivo)
    print(f"Época {epoca:02d}/{MAX_EPOCAS:02d} | Perda Média Treino: {perda_media:.4f} | Acurácia Teste: {acuracia_atual:.2f}%")
    
    # Critério de paragem
    if acuracia_atual >= LIMIAR_ACURACIA:
        print(f"\nMeta atingida com sucesso na época {epoca:02d}: Acurácia ({acuracia_atual:.2f}%) >= {LIMIAR_ACURACIA}%!")
        break

# 6. Gravação dos pesos finais
nome_arquivo_modelo = "modelo_cifar_cnn.pth"
torch.save(modelo.state_dict(), nome_arquivo_modelo)
print(f"Pesos do modelo gravados em: '{nome_arquivo_modelo}'")

Treino a executar na CPU.
A carregar CIFAR-10...

--- INICIANDO TREINO (Máximo: 15 épocas | Meta: >=65.0%) ---
Época 01/15 | Perda Média Treino: 1.4254 | Acurácia Teste: 57.85%
Época 02/15 | Perda Média Treino: 1.0724 | Acurácia Teste: 63.06%
Época 03/15 | Perda Média Treino: 0.9373 | Acurácia Teste: 66.63%

Meta atingida com sucesso na época 03: Acurácia (66.63%) >= 65.0%!
Pesos do modelo gravados em: 'modelo_cifar_cnn.pth'
